In [1]:
%pip install imbalanced-learn
%pip install lightgbm
%pip install numexpr --upgrade

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import glob
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from lightgbm import LGBMClassifier


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)

Matplotlib is building the font cache; this may take a moment.


In [4]:
# full dataset
path = "swb-s3926387-personal-study"
#path = r'C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/CIC-IDS/CSV-2023/MERGED_CSV'
all_files = glob.glob(path + "/*.csv")
li = []
for filename in all_files:
    df = pd.read_csv(filename, encoding='cp1252', index_col=None, header=0)
    li.append(df)
    print("Read Completed for ", filename)
df = pd.concat(li, axis=0, ignore_index=True)
df.describe()
df.head()
df.info()

Read Completed for  swb-s3926387-personal-study/Merged01.csv
Read Completed for  swb-s3926387-personal-study/Merged02.csv
Read Completed for  swb-s3926387-personal-study/Merged03.csv
Read Completed for  swb-s3926387-personal-study/Merged04.csv
Read Completed for  swb-s3926387-personal-study/Merged05.csv
Read Completed for  swb-s3926387-personal-study/Merged06.csv
Read Completed for  swb-s3926387-personal-study/Merged07.csv
Read Completed for  swb-s3926387-personal-study/Merged08.csv
Read Completed for  swb-s3926387-personal-study/Merged09.csv
Read Completed for  swb-s3926387-personal-study/Merged10.csv
Read Completed for  swb-s3926387-personal-study/Merged11.csv
Read Completed for  swb-s3926387-personal-study/Merged12.csv
Read Completed for  swb-s3926387-personal-study/Merged13.csv
Read Completed for  swb-s3926387-personal-study/Merged14.csv
Read Completed for  swb-s3926387-personal-study/Merged15.csv
Read Completed for  swb-s3926387-personal-study/Merged16.csv
Read Completed for  swb-

In [ ]:
delete_cat = ['DDOS-RSTFINFLOOD', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD', 'DDOS-TCP_FLOOD', 'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'RECON-OSSCAN', 'RECON-PORTSCAN']

df = df[~df['Label'].isin(delete_cat)].reset_index(drop=True)

In [4]:
print(df.columns)
print(df["Label"].value_counts())
print(df.shape)

Index(['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate',
       'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
       'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
       'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count',
       'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP',
       'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max',
       'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label'],
      dtype='object')
Label
DDOS-ICMP_FLOOD            6893259
DDOS-UDP_FLOOD             5181027
DDOS-TCP_FLOOD             4306086
DDOS-PSHACK_FLOOD          3920372
DDOS-SYN_FLOOD             3886130
DDOS-RSTFINFLOOD           3872808
DDOS-SYNONYMOUSIP_FLOOD    3445659
DOS-UDP_FLOOD              3177323
DOS-TCP_FLOOD              2558256
DOS-SYN_FLOOD              1942176
BENIGN                     1051373
MIRAI-GREETH_FLOOD          949381
MIRAI-UDPPLAIN              852695
MIRAI-GREIP_FLOOD   

In [5]:
# As shown in feature importances, some low importance features will be deleted
df = df.drop(columns=['SMTP', 'Telnet', 'IRC', 'cwr_flag_number', 'IGMP', 'Number', 'LLC', 'ece_flag_number', 'DHCP', 'SSH', 'Protocol Type', 'IPv'])

Many network intrusion datasets (like CICIoT2023) contain very specific attack names (e.g., 'DDoS-UDP_Flood', 'SqlInjection'). These are too granular for efficient model training and analysis, so we group them into broader categories, such as 'DDoS', 'DoS', 'Recon', etc.

<h3> Why this is useful </h3>
    <li> Reduces label complexity (from 30+ labels to ~6–7 categories).</li>
    <li> Helps balance the dataset more evenly.</li>
    <li> Simplifies the classification problem for your ML model.</li>

<h3> Example: </h3>
<p>If your original data had this:</p>
<P>Label</p>
<ul>
    <li>DDoS-UDP_Flood</li>
    <li>DDoS-ACK_Fragmentation</li>
    <li>SqlInjection</li>
</ul>

<p>It will become:</p>
<p>Label</p>
<ul>
    <li>DDoS</li>
    <li>DDoS</li>
    <li>Web_based</li>
</ul>

In [ ]:
DDoS = ['DDOS-ACK_FRAGMENTATION',     
        'DDOS-UDP_FLOOD',       
        'DDOS-SLOWLORIS',               
        'DDOS-ICMP_FLOOD',     
        'DDOS-PSHACK_FLOOD',        
        'DDOS-HTTP_FLOOD',              
        'DDOS-UDP_FRAGMENTATION',         
        'DDOS-ICMP_FRAGMENTATION']

DoS = ['DOS-UDP_FLOOD',       
       'DOS-HTTP_FLOOD']

Spoofing = ['MITM-ARPSPOOFING',     
            'DNS_SPOOFING']

Brute_force = ['DICTIONARYBRUTEFORCE']

Recon = ['RECON-HOSTDISCOVERY',                
         'RECON-PINGSWEEP',         
         'VULNERABILITYSCAN']

Web_based = ['SQLINJECTION',        
             'BROWSERHIJACKING',         
             'COMMANDINJECTION',       
             'XSS',         
             'BACKDOOR_MALWARE',          
             'UPLOADING_ATTACK']

Mirai = ['MIRAI-GREETH_FLOOD',     
         'MIRAI-UDPPLAIN',     
         'MIRAI-GREIP_FLOOD']



def classify_attacks(data):
    data['Label'].replace(DDoS,'DDoS',inplace=True)
    data['Label'].replace(DoS,'DoS',inplace=True)
    data['Label'].replace(Spoofing,'Spoofing',inplace=True)
    data['Label'].replace(Brute_force,'Brute_Force',inplace=True)
    data['Label'].replace(Recon,'Recon',inplace=True)
    data['Label'].replace(Web_based,'Web_based',inplace=True)
    data['Label'].replace(Mirai,'Mirai',inplace=True)
classify_attacks(df)

print(df["Label"].value_counts())
print(df.shape)

Label
DDoS           32536197
DoS             7746554
Mirai           2521731
BENIGN          1051373
Recon            661121
Spoofing         465937
Web_based         23799
Brute_Force       12522
Name: count, dtype: int64
(45019243, 28)


In [7]:
df

,Header_Length,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ack_count,syn_count,...,ICMP,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Variance,Label
0,19.92,63.36,25893.962218,0.0,0.00,0.0,0.99,0.99,99.0,0.0,...,0.00,6421.0,60.0,481.0,64.21,42.100000,64.21,0.000039,1772.410000,DDoS
1,0.00,64.00,3703.841331,0.0,0.00,0.0,0.00,0.00,0.0,0.0,...,0.01,57320.0,98.0,578.0,573.20,48.000000,573.20,0.000271,2304.000000,Mirai
2,7.92,65.91,19673.095685,0.0,0.00,0.0,0.00,0.00,0.0,0.0,...,0.01,6010.0,60.0,70.0,60.10,1.000000,60.10,0.000057,1.000000,DoS
3,20.40,110.50,261.664826,0.1,0.00,0.3,0.20,0.40,4.0,0.0,...,0.00,2223.0,54.0,1500.0,222.30,451.596686,222.30,0.004766,203939.566667,Spoofing
4,0.32,63.96,28944.199848,0.0,0.00,0.0,0.00,0.01,1.0,0.0,...,0.99,6006.0,60.0,66.0,60.06,0.600000,60.06,0.000035,0.360000,DDoS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45019238,20.00,64.00,33267.005076,0.0,1.00,0.0,0.00,0.00,0.0,100.0,...,0.00,6000.0,60.0,60.0,60.00,0.000000,60.00,0.000030,0.000000,DDoS
45019239,20.00,64.00,8280.792087,0.0,1.00,0.0,0.00,0.00,0.0,100.0,...,0.00,6000.0,60.0,60.0,60.00,0.000000,60.00,0.000121,0.000000,DDoS
45019240,20.00,64.00,31939.567469,0.0,0.00,0.0,0.00,0.00,0.0,0.0,...,0.00,6000.0,60.0,60.0,60.00,0.000000,60.00,0.000033,0.000000,DDoS
45019241,8.24,65.55,3334.661589,0.0,0.01,0.0,0.00,0.00,0.0,1.0,...,0.01,6075.0,60.0,90.0,60.75,3.934758,60.75,0.000300,15.482323,DDoS


Non-numerical data is incomprehensible to machine learning modules. To avoid issues later on, the data should be arranged numerically. The answer to this problem is to convert all text values to numerical form.

<p>data_clean = df.drop_duplicates(keep='first', inplace = True)</p>
<p>data_clean = data_clean.replace([np.inf, -np.inf], np.nan, inplace=True)</p>
<p>data_clean = data_clean.dropna(inplace=True).reset_index(drop=True) </p>
<h3>AttributeError: 'NoneType' object has no attribute 'replace'</h3>
<p>The issue is that inplace=True makes the method return None, so data_clean becomes None.
<h2>Why This Matters:</h2>
<li>inplace=True means the operation modifies the original DataFrame and returns None.
<li>So doing data_clean = df.drop_duplicates(inplace=True) sets data_clean to None.

In [6]:
df.drop_duplicates(keep='first', inplace = True)
df.replace([np.inf, -np.inf], np.nan, inplace=True) # Replace inf and -inf with NaN, then drop the resulting NaNs
df.dropna(inplace=True)    # dropna() removes any rows that contain NaN (missing) values; reset_index() resets the row index after dropping, so it's clean and continuous.
df.reset_index(drop=True, inplace=True)

print("Read {} rows.".format(len(df)))

# data_clean.columns          # This just lists the column names in the data_clean DataFrame—helpful for debugging or inspection.   
print(df.shape)
print(df['Label'].value_counts())   # Shows how many samples exist for each class (now as numbers). This is useful to check for class imbalance.

Read 20984394 rows.
(20984394, 28)
Label
DDOS-UDP_FLOOD             1964066
DDOS-ICMP_FLOOD            1907542
DOS-UDP_FLOOD              1851252
DDOS-SYN_FLOOD             1761479
DDOS-PSHACK_FLOOD          1638478
DDOS-TCP_FLOOD             1556620
DDOS-RSTFINFLOOD           1251851
DDOS-SYNONYMOUSIP_FLOOD    1167941
DOS-SYN_FLOOD              1137758
DOS-TCP_FLOOD              1118673
BENIGN                     1047308
MIRAI-GREETH_FLOOD          897595
MIRAI-UDPPLAIN              767297
MIRAI-GREIP_FLOOD           694290
DDOS-ICMP_FRAGMENTATION     431149
VULNERABILITYSCAN           356322
DDOS-UDP_FRAGMENTATION      273392
DDOS-ACK_FRAGMENTATION      271535
MITM-ARPSPOOFING            269044
DNS_SPOOFING                167017
RECON-HOSTDISCOVERY         127867
RECON-OSSCAN                 92311
RECON-PORTSCAN               76782
DOS-HTTP_FLOOD               68441
DDOS-HTTP_FLOOD              27597
DDOS-SLOWLORIS               22399
DICTIONARYBRUTEFORCE         12520
BROWSERHIJACKI

In [ ]:
# Before correlation filter
# Select only numeric columns for correlation
numeric_data = df.select_dtypes(include=['number'])   # Or df.drop(columns=['Label'])

plt.figure(figsize=(25, 13))
sns.heatmap(numeric_data.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [7]:
# Delete the high corralation features
df = df.drop(columns=['fin_count', 'fin_flag_number', 'rst_count', 'rst_flag_number', 'Tot size', 'syn_count', 'ack_count'])

In [ ]:
# After correlation filter
# Select only numeric columns for correlation
numeric_data = df.select_dtypes(include=['number'])   # Or df.drop(columns=['Label'])

plt.figure(figsize=(25, 13))
sns.heatmap(numeric_data.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [ ]:
columns_name = df.drop(columns=['Label']).columns

In [ ]:
# data_np = data_clean.to_numpy(dtype="float32")      # Converts the entire DataFrame into a NumPy array with float32 data type—faster and more efficient for ML model input.
# #drop inf values
# data_np = data_np[~np.isinf(data_np).any(axis=1)]   # Removes any rows that have infinite values (inf) across any column. This ensures clean numeric data for training

In [ ]:
# Dataset check
# for i in range(100):
#     print(data_clean['index'][i])


In [8]:
rows = df.shape[0]
limit = round(rows*50/100) # split the dataset into 2 part: 80% for train and val; 20% to test
train_val = df[1:limit] 	

test = df[limit:rows]

In [9]:
print(train_val.shape)
# print(train_val['Label'].value_counts())

(10492196, 21)


In [10]:
# X = train_val[:, :-1] 
# Y = train_val[:, -1]

In [11]:
X = train_val.drop(columns=['Label'])
Y = train_val['Label']

In [12]:
print(Y.shape)
print(X.shape)

(10492196,)
(10492196, 20)


In [15]:
# This block is created to keep track with the under or over sampling

label_series = pd.Series(Y) # data_np is numPy type so it need to revert back to DataFrame to use value_counts
sample_limit = label_series.value_counts()[13]
print(label_series.value_counts())
print(sample_limit)

Label
DDOS-ICMP_FLOOD            1009704
DDOS-UDP_FLOOD              994726
DOS-UDP_FLOOD               912231
DDOS-SYN_FLOOD              886021
DDOS-PSHACK_FLOOD           843538
DDOS-TCP_FLOOD              796902
DDOS-RSTFINFLOOD            660243
DDOS-SYNONYMOUSIP_FLOOD     617103
DOS-TCP_FLOOD               567594
DOS-SYN_FLOOD               563592
BENIGN                      486128
MIRAI-GREETH_FLOOD          425104
MIRAI-UDPPLAIN              370258
MIRAI-GREIP_FLOOD           326263
DDOS-ICMP_FRAGMENTATION     200238
VULNERABILITYSCAN           165035
DDOS-UDP_FRAGMENTATION      126764
DDOS-ACK_FRAGMENTATION      126199
MITM-ARPSPOOFING            125467
DNS_SPOOFING                 77772
RECON-HOSTDISCOVERY          59692
RECON-OSSCAN                 42918
RECON-PORTSCAN               36031
DOS-HTTP_FLOOD               31651
DDOS-HTTP_FLOOD              12777
DDOS-SLOWLORIS               10424
DICTIONARYBRUTEFORCE          5809
BROWSERHIJACKING              2571
SQLINJECTION  

In [16]:
# Unbalance handling

# Under sampling
# Grouped labels
#undersample = RandomUnderSampler(sampling_strategy={'DDoS':sample_limit, 'DoS':sample_limit}) # if the strategy ='all' - normal undersample; if it a dict then the key is the name of the label and the value is 
                                                                                    # the target undersample - in this case '2' is the name of the 'DDOS' after encoded

# Separated labels
undersample = RandomUnderSampler(sampling_strategy={'DDOS-ICMP_FLOOD':sample_limit, 'DDOS-UDP_FLOOD':sample_limit, 'DOS-UDP_FLOOD':sample_limit, 'DDOS-SYN_FLOOD':sample_limit, 'DDOS-PSHACK_FLOOD':sample_limit, 'DDOS-TCP_FLOOD':sample_limit, 'DDOS-RSTFINFLOOD':sample_limit, 'DDOS-SYNONYMOUSIP_FLOOD':sample_limit, 'DOS-TCP_FLOOD':sample_limit, 'DOS-SYN_FLOOD':sample_limit, 'MIRAI-GREETH_FLOOD':sample_limit, 'MIRAI-UDPPLAIN':sample_limit})

X_under, y_under = undersample.fit_resample(X, Y)

# Over sampling
sm = SMOTE()
X_sm, Y_sm = sm.fit_resample(X_under, y_under)

print(Y_sm.shape)
print(X_sm.shape)

(16528352,)
(16528352, 20)


In [17]:
# Under sampling only
# undersample = RandomUnderSampler(sampling_strategy='all')   # if the strategy ='all' - normal undersample; if it a dict then the key is the name of the label and the value is 
#                                                             # the target undersample - in this case '2' is the name of the 'DDOS' after encoded
# X_sm, Y_sm = undersample.fit_resample(X, Y)

In [18]:
# This block is created to keep track with the under or over sampling

# labels = Y_sm.astype(int) # data_np default as float, float can't work with the inverse transform

# decoded_labels = labelencoder.inverse_transform(labels) # Inverse transform the whole array
#                                                         # inverse_transform() accepts a NumPy array, so it internally uses vectorized operations

# label_series = pd.Series(decoded_labels) # data_np is numPy type so it need to revert back to DataFrame to use value_counts
# print(label_series.value_counts())

label_series = pd.Series(Y_sm) # data_np is numPy type so it need to revert back to DataFrame to use value_counts
print(label_series.value_counts())

Label
BACKDOOR_MALWARE           486128
BENIGN                     486128
BROWSERHIJACKING           486128
COMMANDINJECTION           486128
DDOS-ACK_FRAGMENTATION     486128
DDOS-HTTP_FLOOD            486128
DDOS-ICMP_FLOOD            486128
DDOS-ICMP_FRAGMENTATION    486128
DDOS-PSHACK_FLOOD          486128
DDOS-RSTFINFLOOD           486128
DDOS-SLOWLORIS             486128
DDOS-SYNONYMOUSIP_FLOOD    486128
DDOS-SYN_FLOOD             486128
DDOS-TCP_FLOOD             486128
DDOS-UDP_FLOOD             486128
DDOS-UDP_FRAGMENTATION     486128
DICTIONARYBRUTEFORCE       486128
DNS_SPOOFING               486128
DOS-HTTP_FLOOD             486128
DOS-SYN_FLOOD              486128
DOS-TCP_FLOOD              486128
DOS-UDP_FLOOD              486128
MIRAI-GREETH_FLOOD         486128
MIRAI-GREIP_FLOOD          486128
MIRAI-UDPPLAIN             486128
MITM-ARPSPOOFING           486128
RECON-HOSTDISCOVERY        486128
RECON-OSSCAN               486128
RECON-PINGSWEEP            486128
RECON-PO

In [19]:
X_test = test.drop(columns=['Label'])
Y_test = test['Label']

What it does:
This scales (standardizes) your feature matrix X.
Specifically:
<ul>
    <li>StandardScaler() creates an object that standardizes the features.
    <li>Standardization means: for each feature/column in X, it will subtract the mean and divide by the standard deviation.
</ul>
Mathematically for each feature:

𝑋_scaled = (𝑋 − mean(𝑋)) / std(𝑋)
 
As a result:
<ul>
    <li>The new mean of each feature = 0.
    <li>The new standard deviation of each feature = 1.
</ul>

Why StandardScale important?
<ul>
    <li>Machine Learning algorithms (especially neural networks) work better when input features are on the same scale.
    <li>If features have very different scales (e.g., some from 0–1, some from 0–10,000), the model can struggle to converge or be biased toward features with larger numbers.
    <li>Standardization makes gradient descent faster and model training more stable.
</ul>

In simple terms:
<ul>
    <li>Before: your features might look like [50, 10000, 3.2, 0.001]
    <li>After StandardScaler: numbers become something like [0.12, 2.5, -1.0, -0.3]
    <li>Now all features are centered around 0 and have similar ranges, which helps the model learn better!
</ul>

In [20]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sm)

In [21]:
X_test_scaled = scaler.fit_transform(X_test)

In [22]:
# Split the data set into training and testing
X_train, X_val, Y_train, Y_val = train_test_split(
    X_scaled, Y_sm, test_size=0.25, random_state=33, shuffle=True)

<h2>LightGBM<h2>

<h4>Hold-out method for LightGBM model<h4>

grid.best_params_ is a dictionary (e.g. {'n_estimators': 100, 'max_depth': 10, ...}), and passing it as a single positional argument will raise an error because RandomForestClassifier() expects keyword arguments.

You should unpack the dictionary using ** like this: RandomForestClassifier(**grid.best_params_)

In [ ]:
# Train model
lgbm_model = LGBMClassifier(
    # device='gpu',
    objective='multiclass',
    num_class=8,                     # Number of output classes
    boosting_type='gbdt',            # Standard gradient boosting
    n_estimators=1000,               # More trees for better learning
    learning_rate=0.01,              # Lower LR with more trees = stable learning
    max_depth=16,                    # Controls tree complexity
    num_leaves=128,                  # More leaves = more complex trees (2^max_depth rule of thumb)
    min_child_samples=20,            # Minimum samples per leaf
    subsample=1.0,                   # Row sampling for each tree (a.k.a. bagging_fraction)
    subsample_freq=1,                # Perform subsample every iteration
    colsample_bytree=0.8,            # Feature sampling (a.k.a. feature_fraction)
    max_bin=255,                     # Histogram bins (increase if features are continuous-heavy)
    n_jobs=-1,                       # Use number of CPU cores (-1 = all cores)
    random_state=42,
    verbose=10000                      # Clean logging
)

history = lgbm_model.fit(X_train, Y_train)

y_val_pred_lgbm = lgbm_model.predict(X_val)
y_train_pred_lgbm = lgbm_model.predict(X_train)

[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.727257
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.109089
[LightGBM] [Debug] init for col-wise cost 0.209912 seconds, init for row-wise cost 0.521313 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.367786 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of data points in the train set: 12396264, number of used features: 20
[LightGBM] [Info] Start training from score -3.526599
[LightGBM] [Info] Start training from score -3.526166
[LightGBM] [Info] Start training from score -3.526166
[LightGBM] [Info] Start training from score -3.524522
[LightGBM] [Info] Start training from score -3.527648
[LightGBM] [Info] Start training from score -3.526687
[LightGBM] [Info] 

<h4>Validate Hold-out Model<h4>

In [ ]:
# print('Accuracy score: %.2f' % accuracy_score(Y_val, y_pred_frst))
# print('Precision score: %.2f' % precision_score(Y_val, y_pred_frst, average='weighted'))
# print('Recall score: %.2f' % recall_score(Y_val, y_pred_frst, average='weighted'))
# print('F1 score: %.2f' % f1_score(Y_val, y_pred_frst, average='weighted'))
print("VAL SCORE")
print(classification_report(Y_val, y_val_pred_lgbm))
print("\n"+"TRAIN SCORE")
print(classification_report(Y_train, y_train_pred_lgbm))

In [ ]:
# Confusion matrix
cm = confusion_matrix(Y_val, y_val_pred_lgbm)

# Extract sorted unique labels from Y_val
class_labels = sorted(Y_val.unique())

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(xticks_rotation=45)

In [ ]:
# Check feature importance
feat_importances = pd.Series(lgbm_model.feature_importances_, index=columns_name)
top_features = feat_importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10,6))
sns.barplot(x=top_features, y=top_features.index)
plt.title("Ranking Feature Importances in Random Forest")
plt.show()

In [ ]:
for i in range(100):
    sample = X_test.iloc[[i]]  # Make it 2D by using double brackets
    pred = lgbm_model.predict(sample)
    actual = Y_test.iloc[i]

    print(f"Sample {i}: Predicted = {pred[0]}, Actual = {actual}")


In [ ]:
# Save the model
with open('C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/Models/CIC_models/lightgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)

In [ ]:
# Load the model
with open('C:/Users/ADMIN/Desktop/code_folders/Capstone_Project/Models/CIC_models/lightgbm_model.pkl', 'rb') as f:
    imported_model = pickle.load(f)

In [29]:
y_val_pred_lgbm = imported_model.predict(X_val)
print(classification_report(Y_val, y_val_pred_lgbm))

c:\Users\ADMIN\anaconda3\envs\thinh\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

      BENIGN       0.89      0.33      0.49    281292
 Brute_Force       0.75      0.87      0.80    281182
        DDoS       0.94      0.46      0.61    280871
         DoS       0.64      0.97      0.77    280406
       Mirai       1.00      1.00      1.00    280678
       Recon       0.61      0.71      0.66    281003
    Spoofing       0.92      0.82      0.87    280956
   Web_based       0.64      0.90      0.75    281030

    accuracy                           0.76   2247418
   macro avg       0.80      0.76      0.74   2247418
weighted avg       0.80      0.76      0.74   2247418



In [28]:
test_label_series = pd.Series(Y_test) # data_np is numPy type so it need to revert back to DataFrame to use value_counts
test_sample_limit = test_label_series.value_counts()[4]
print(test_label_series.value_counts())
print(test_sample_limit)

Label
DDoS           5912393
DoS            2097324
Mirai          1235473
BENIGN          560264
Recon           349264
Spoofing        232251
Web_based        12685
Brute_Force       6700
Name: count, dtype: int64
349264


In [29]:
# Under sampling
test_undersample = RandomUnderSampler(sampling_strategy={'DDoS':test_sample_limit, 'DoS':test_sample_limit, 'Mirai':test_sample_limit, 'BENIGN':test_sample_limit}) # if the strategy ='all' - normal undersample; if it a dict then the key is the name of the label and the value is 
                                                                                    # the target undersample - in this case '2' is the name of the 'DDOS' after encoded

X_test_under, y_test_under = test_undersample.fit_resample(X_test_scaled, Y_test)

# Over sampling
test_sm = SMOTE()
X_test_sm, Y_test_sm = test_sm.fit_resample(X_test_under, y_test_under)

print(Y_test_sm.shape)
print(X_test_sm.shape)

(2794112,)
(2794112, 20)


In [30]:
test_label_series = pd.Series(Y_test_sm) # data_np is numPy type so it need to revert back to DataFrame to use value_counts
print(test_label_series.value_counts())

Label
BENIGN         349264
Brute_Force    349264
DDoS           349264
DoS            349264
Mirai          349264
Recon          349264
Spoofing       349264
Web_based      349264
Name: count, dtype: int64


In [31]:
test_pred = imported_model.predict(X_test_sm)
print(classification_report(Y_test_sm, test_pred))

c:\Users\ADMIN\anaconda3\envs\thinh\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

      BENIGN       0.40      0.40      0.40    349264
 Brute_Force       0.00      0.00      0.00    349264
        DDoS       0.96      0.09      0.16    349264
         DoS       0.60      0.01      0.02    349264
       Mirai       0.95      0.99      0.97    349264
       Recon       0.21      0.84      0.33    349264
    Spoofing       0.39      0.40      0.40    349264
   Web_based       0.05      0.04      0.04    349264

    accuracy                           0.35   2794112
   macro avg       0.45      0.35      0.29   2794112
weighted avg       0.45      0.35      0.29   2794112

